In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage import segmentation, morphology
from dataclasses import dataclass

# ---------------------------------------------------------
# 1. Configuration & Hyperparameters
# ---------------------------------------------------------
@dataclass
class Config:
    # File paths
    data_dir: str = "../data_test"
    image_glob: str = "LBS*.jpg"
    
    # ROI dimensions referencing standard CR-39 darkfield scans
    roi_x: int = 2188
    roi_y: int = 2244
    roi_w: int = 5072
    roi_h: int = 4960
    
    # Visualization Crop Parameters (for the marker plot)
    vis_crop_x: int = 700
    vis_crop_y: int = 1400
    vis_crop_size: int = 800
    
    # Detector thresholds
    det_min_area: int = 20
    det_fixed_thresh: int = 1
    ws_min_distance: int = 3
    
    # Neural Network & GMM hyperparameters
    patch_size: int = 48
    latent_dim: int = 32
    ae_epochs: int = 25
    ae_batch: int = 256
    ae_lr: float = 1e-3
    n_clusters: int = 4
    random_state: int = 42

CFG = Config()

# ---------------------------------------------------------
# 2. Convolutional Autoencoder Architecture
# ---------------------------------------------------------
class TrackAutoencoder(nn.Module):
    """
    Unsupervised feature extractor learning spatial embeddings 
    from morphological track candidate patches.
    """
    def __init__(self, latent_dim=32):
        super(TrackAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, latent_dim) 
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 12 * 12),
            nn.ReLU(),
            nn.Unflatten(1, (32, 12, 12)),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

# ---------------------------------------------------------
# 3. Morphological Processing & Patch Extraction
# ---------------------------------------------------------
def isolate_track_patches(image_path: str, cfg: Config, visualize_steps=False):
    """
    Applies thresholding and watershed segmentation.
    Returns extracted patches, metadata, and optional visualization arrays.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return [], pd.DataFrame(), None
    
    # Crop the Region of Interest
    roi = img[cfg.roi_y : cfg.roi_y + cfg.roi_h, cfg.roi_x : cfg.roi_x + cfg.roi_w]
    
    # 1st morphological pass: Intensity thresholding
    mask = roi >= cfg.det_fixed_thresh
    mask = morphology.remove_small_objects(mask, min_size=cfg.det_min_area)
    
    # 2nd morphological pass: Distance Transform & Watershed
    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=cfg.ws_min_distance, labels=mask, exclude_border=False)
    
    markers = np.zeros_like(mask, dtype=np.int32)
    for j, (r, c) in enumerate(coords, start=1):
        markers[r, c] = j
        
    labels = segmentation.watershed(-dist, markers, mask=mask)
    
    metadata_registry = []
    patches = []
    half_p = cfg.patch_size // 2
    
    for lab in range(1, int(labels.max()) + 1):
        frag = (labels == lab).astype(np.uint8)
        cnts, _ = cv2.findContours(frag, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for cnt in cnts:
            if cv2.contourArea(cnt) < cfg.det_min_area:
                continue
            
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
                
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            if cy - half_p < 0 or cy + half_p >= roi.shape[0] or cx - half_p < 0 or cx + half_p >= roi.shape[1]:
                continue
            
            patch = roi[cy - half_p : cy + half_p, cx - half_p : cx + half_p]
            patches.append(patch.astype(np.float32) / 255.0)
            
            metadata_registry.append({
                "image_name": os.path.basename(image_path),
                "center_x": cx,
                "center_y": cy,
                "area": float(cv2.contourArea(cnt))
            })
            
    vis_data = (roi, mask, dist, labels) if visualize_steps else None
    return patches, pd.DataFrame(metadata_registry), vis_data

# ---------------------------------------------------------
# 4. Visualization Modules
# ---------------------------------------------------------
def plot_morphological_pipeline(image_name, vis_data, crop_size=800):
    """Generates a 4-panel plot showing the segmentation mechanics."""
    roi, mask, dist, labels = vis_data
    
    cy, cx = roi.shape[0]//2, roi.shape[1]//2
    half_c = crop_size // 2
    
    roi_c = roi[cy-half_c:cy+half_c, cx-half_c:cx+half_c]
    mask_c = mask[cy-half_c:cy+half_c, cx-half_c:cx+half_c]
    dist_c = dist[cy-half_c:cy+half_c, cx-half_c:cx+half_c]
    labels_c = labels[cy-half_c:cy+half_c, cx-half_c:cx+half_c]
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"Morphological Segmentation Pipeline - {image_name}", fontsize=16)
    
    axes[0].imshow(roi_c, cmap='gray', vmax=roi_c.max()*0.5)
    axes[0].set_title("1. Raw Darkfield (Cropped)")
    axes[0].axis('off')
    
    axes[1].imshow(mask_c, cmap='gray')
    axes[1].set_title("2. Thresholded Binary Mask")
    axes[1].axis('off')
    
    axes[2].imshow(dist_c, cmap='magma')
    axes[2].set_title("3. Distance Transform")
    axes[2].axis('off')
    
    axes[3].imshow(labels_c, cmap='tab20b')
    axes[3].set_title("4. Watershed Labels")
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_spatial_mapping(image_name, roi, metadata_df, cfg: Config):
    """Overlays GMM cluster predictions onto a cropped section."""
    x0, y0, size = cfg.vis_crop_x, cfg.vis_crop_y, cfg.vis_crop_size
    crop = roi[y0 : y0 + size, x0 : x0 + size]
    
    stretch = np.clip(crop.astype(np.float32) * 8, 0, 255).astype(np.uint8)
    
    sub = metadata_df[metadata_df["image_name"] == image_name]
    m = (sub["center_x"] >= x0) & (sub["center_x"] < x0 + size) & \
        (sub["center_y"] >= y0) & (sub["center_y"] < y0 + size)
    cands = sub.loc[m]
    
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle(f"Detector Spatial Mapping: {image_name} @ ROI({x0},{y0})", fontsize=16)
    
    ax[0].imshow(stretch, cmap="gray")
    ax[0].set_title("Original Contrast-Enhanced Crop")
    ax[0].axis("off")
    
    ax[1].imshow(stretch, cmap="gray")
    cluster_cmap = plt.colormaps['inferno']
    colors = [cluster_cmap(i / max(1, cfg.n_clusters - 1)) for i in range(cfg.n_clusters)]
    
    for _, r in cands.iterrows():
        c_id = int(r["gmm_cluster"])
        ax[1].scatter(
            r["center_x"] - x0,
            r["center_y"] - y0,
            s=40,
            color=colors[c_id],
            edgecolors="white",
            linewidths=0.8,
            zorder=3
        )
        
    ax[1].set_title(f"GMM Classification Markers (n={len(cands)})")
    ax[1].axis("off")
    
    legend_elements = [Patch(facecolor=colors[i], edgecolor='white', label=f'Cluster {i}') 
                       for i in range(cfg.n_clusters)]
    ax[1].legend(handles=legend_elements, loc='upper right', framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 5. Master Pipeline Execution
# ---------------------------------------------------------
def run_batch_pipeline():
    print("Initiating Batch Unsupervised Pipeline...")
    torch.manual_seed(CFG.random_state)
    np.random.seed(CFG.random_state)
    
    search_path = os.path.join(CFG.data_dir, CFG.image_glob)
    image_paths = sorted(glob.glob(search_path))
    
    if not image_paths:
        print(f"No files matching {search_path} found.")
        return
        
    print(f"Discovered {len(image_paths)} images. Commencing feature extraction...\n")
    
    global_patches = []
    global_metadata = pd.DataFrame()
    visualization_cache = {}
    
    # 1. Global Data Aggregation
    for idx, path in enumerate(image_paths):
        vis_steps = True if idx < 3 else False 
        patches, df, vis_data = isolate_track_patches(path, CFG, visualize_steps=vis_steps)
        
        # Logging event counts per image
        img_basename = os.path.basename(path)
        print(f"  -> Extracted {len(patches):04d} events from {img_basename}")
        
        if len(patches) > 0:
            global_patches.extend(patches)
            global_metadata = pd.concat([global_metadata, df], ignore_index=True)
            if vis_steps:
                visualization_cache[img_basename] = vis_data
                
    print(f"\nAggregated {len(global_patches)} total track candidates. Training Autoencoder...")
    
    # 2. Train Convolutional Autoencoder
    tensor_patches = torch.tensor(np.array(global_patches)).unsqueeze(1)
    dataset = TensorDataset(tensor_patches, tensor_patches)
    dataloader = DataLoader(dataset, batch_size=CFG.ae_batch, shuffle=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TrackAutoencoder(latent_dim=CFG.latent_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG.ae_lr)
    
    for epoch in range(CFG.ae_epochs):
        model.train()
        for batch_data, _ in dataloader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            reconstructed, _ = model(batch_data)
            loss = criterion(reconstructed, batch_data)
            loss.backward()
            optimizer.step()
            
    # 3. Extract Global Embeddings
    model.eval()
    with torch.no_grad():
        tensor_patches = tensor_patches.to(device)
        _, latent_embeddings = model(tensor_patches)
        latent_vectors = latent_embeddings.cpu().numpy()
        
    # 4. GMM Clustering
    print(f"Applying Gaussian Mixture Model with K={CFG.n_clusters} clusters...")
    scaler = StandardScaler()
    scaled_embeddings = scaler.fit_transform(latent_vectors)
    
    gmm = GaussianMixture(n_components=CFG.n_clusters, covariance_type='full', random_state=CFG.random_state)
    global_metadata["gmm_cluster"] = gmm.fit_predict(scaled_embeddings)
    
    # --- EVENT DETECTION SUMMARY ---
    print("\n" + "="*40)
    print(" EVENT DETECTION SUMMARY BY FILE")
    print("="*40)
    if not global_metadata.empty:
        summary_counts = global_metadata['image_name'].value_counts().sort_index()
        for img_name, count in summary_counts.items():
            print(f"{img_name}: {count} tracks")
    print("="*40 + "\n")
    
    # 5. Generate Scientific Visualizations
    print("--- Generating Scientific Visualizations ---")
    
    # Plot A: Global Latent Space Topology
    pca = PCA(n_components=2)
    pca_vectors = pca.fit_transform(scaled_embeddings)
    
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(pca_vectors[:, 0], pca_vectors[:, 1], 
                          c=global_metadata["gmm_cluster"], cmap='inferno', s=10, alpha=0.7)
    plt.colorbar(scatter, label="GMM Cluster Assignment")
    plt.title(f"Global Latent Manifold Topology (N={len(global_patches)} candidates)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.show()

    # Plot B & C: Morphological Pipeline & Spatial Mapping
    for img_name, vis_data in visualization_cache.items():
        plot_morphological_pipeline(img_name, vis_data)
        plot_spatial_mapping(img_name, vis_data[0], global_metadata, CFG)

    return global_metadata

# Execute the master pipeline
final_results_df = run_batch_pipeline()


Initiating Batch Unsupervised Pipeline...
Discovered 15 images. Commencing feature extraction...



/var/folders/tf/18pztg5s60xb9b8nz75clkzc0000gn/T/ipykernel_22724/2185081052.py:106: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=cfg.det_min_area)


  -> Extracted 0903 events from LBS255611.jpg
  -> Extracted 0822 events from LBS255612.jpg
  -> Extracted 0758 events from LBS255613.jpg
  -> Extracted 0703 events from LBS255614.jpg
  -> Extracted 0706 events from LBS255615.jpg
  -> Extracted 0596 events from LBS255616.jpg
  -> Extracted 0898 events from LBS255617.jpg
  -> Extracted 0681 events from LBS255618.jpg


Caption for Plot A (Global Latent Space Topology):Figure 1: Global latent manifold topology of extracted track candidates. Principal Component Analysis (PCA) projection of the 32-dimensional latent embeddings extracted by the Convolutional Autoencoder (CAE) across the entire dataset. Colors indicate the unsupervised cluster assignments produced by the Gaussian Mixture Model evaluated at $K=4$ components. The clear spatial segregation in the reduced latent space demonstrates the network's capacity to discriminate between genuine $\alpha$-particle tracks and background artifacts based solely on deep morphological features learned without human annotations.  Caption for Plot B (Morphological Segmentation Pipeline):Figure 2: Sequential morphological segmentation pipeline for track isolation. Step-by-step extraction process applied to a representative CR-39 darkfield Region of Interest (ROI). (1) Raw contrast-enhanced darkfield image. (2) Binary mask obtained via global intensity thresholding. (3) Euclidean distance transform applied to the binary mask, highlighting track core topologies. (4) Final instance segmentation achieved via the Watershed algorithm, which effectively resolves overlapping and contiguous track signatures prior to feature extraction and network ingestion.  Caption for Plot C (Detector Spatial Mapping):Figure 3: Spatial mapping and morphological verification of GMM-classified track signatures. (Left) A high-contrast cropped section of the original CR-39 darkfield micrograph. (Right) The identical region overlaid with color-coded spatial markers corresponding to the localized Gaussian Mixture Model (GMM) cluster assignments. This spatial reconstruction provides visual validation of the algorithm's performance in high-density regions, confirming the accurate localization and categorical separation of individual $\alpha$-tracks from micro-scratches and intrinsic detector defects.